# Transfer Learning in Practice: Fine-tuning BERT

**Objective**: Practically demonstrate the difference between:
1. Training **from scratch** (random embeddings)
2. **Feature extraction** (freeze BERT, train only classifier)
3. **Fine-tuning** (unfreeze BERT, re-train everything)

**Task**: Sentiment Analysis su IMDb movie reviews

**Dataset**: IMDb 50k reviews (25k train, 25k test)

---

## Setup & Imports

In [16]:
# install the libraries if not installed
!pip install transformers datasets torch scikit-learn matplotlib seaborn

In [17]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import (
    BertTokenizer,
    BertModel,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import time


In [18]:
from torch.optim import AdamW # Corrected import for AdamW

In [19]:

# Set seed per riproducibilità
torch.manual_seed(42)
np.random.seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Load Dataset

We use a **subset** of the dataset IMDb to speed up training:
- **Train**: 2000 samples (1000 positive, 1000 negative)
- **Test**: 500 samples (250 positive, 250 negative)

In [20]:
# Carica dataset IMDb
dataset = load_dataset('imdb')

# Usa un subset per velocità (puoi aumentare per risultati migliori)
train_size = 2000
test_size = 500

train_dataset = dataset['train'].shuffle(seed=42).select(range(train_size))
test_dataset = dataset['test'].shuffle(seed=42).select(range(test_size))

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"\nExample:")
print(f"Text: {train_dataset[0]['text'][:200]}...")
print(f"Label: {train_dataset[0]['label']} (0=negative, 1=positive)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train samples: 2000
Test samples: 500

Example:
Text: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. F...
Label: 1 (0=negative, 1=positive)


## Tokenization

We use BERT tokenizer to convert text into token IDs.

In [21]:
# Carica tokenizer BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128,  # Riduciamo per velocità
        return_tensors='pt'
    )

# Tokenizza dataset
print("Tokenizing...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Formato per PyTorch
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("✅ Tokenization complete!")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizing...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

✅ Tokenization complete!


## Helper Functions

In [22]:
def train_epoch(model, dataloader, optimizer, scheduler, device):
    """Train per un epoch"""
    model.train()
    total_loss = 0

    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    """Valuta il modello"""
    model.eval()
    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(true_labels, predictions)
    return accuracy, predictions, true_labels

---

# Experiment 1: Training FROM SCRATCH

**Approach**: Initialize BERT with random weights (no pre-training)

**Expected result**: Poor performance (no prior knowledge)

In [ ]:
print("=" * 60)
print("EXPERIMENT 1: Training FROM SCRATCH")
print("=" * 60)

# Carica modello con pesi RANDOM
from transformers import BertConfig
config = BertConfig.from_pretrained('bert-base-uncased', num_labels=2)
model_scratch = BertForSequenceClassification(config).to(device)

print(f"\n🔢 Total parameters: {sum(p.numel() for p in model_scratch.parameters()):,}")
print(f"🔢 Trainable parameters: {sum(p.numel() for p in model_scratch.parameters() if p.requires_grad):,}")

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Optimizer & Scheduler
epochs = 3
optimizer = AdamW(model_scratch.parameters(), lr=2e-5)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# Training
start_time = time.time()
scratch_losses = []

for epoch in range(epochs):
    print(f"\n📍 Epoch {epoch + 1}/{epochs}")
    loss = train_epoch(model_scratch, train_loader, optimizer, scheduler, device)
    scratch_losses.append(loss)
    print(f"  Average loss: {loss:.4f}")

scratch_time = time.time() - start_time

# Evaluate
scratch_acc, _, _ = evaluate(model_scratch, test_loader, device)

print(f"\n✅ FROM SCRATCH Results:")
print(f"  Accuracy: {scratch_acc:.4f}")
print(f"  Training time: {scratch_time:.2f}s")

EXPERIMENT 1: Training FROM SCRATCH


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]


🔢 Total parameters: 109,483,778
🔢 Trainable parameters: 109,483,778

📍 Epoch 1/3


Training:   0%|          | 0/125 [00:00<?, ?it/s]

---

# Experiment 2: FEATURE EXTRACTION

**Approach**:
- Load pre-trained BERT
- **FREEZE** all BERT layers
- Train only the final classifier

**Expected result**: Better than scratch (uses pre-trained features)

In [ ]:
print("=" * 60)
print("EXPERIMENT 2: FEATURE EXTRACTION")
print("=" * 60)

# Carica modello PRE-TRAINED
model_feature = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
).to(device)

# FREEZE tutti i layer di BERT (no training)
for param in model_feature.bert.parameters():
    param.requires_grad = False

# Solo il classifier è trainable
print(f"\n🔢 Total parameters: {sum(p.numel() for p in model_feature.parameters()):,}")
print(f"🔢 Trainable parameters: {sum(p.numel() for p in model_feature.parameters() if p.requires_grad):,}")
print(f"   → Only {sum(p.numel() for p in model_feature.parameters() if p.requires_grad) / sum(p.numel() for p in model_feature.parameters()) * 100:.2f}% trainable!")

# Optimizer (solo parametri trainable)
optimizer = AdamW(filter(lambda p: p.requires_grad, model_feature.parameters()), lr=2e-5)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# Training
start_time = time.time()
feature_losses = []

for epoch in range(epochs):
    print(f"\n📍 Epoch {epoch + 1}/{epochs}")
    loss = train_epoch(model_feature, train_loader, optimizer, scheduler, device)
    feature_losses.append(loss)
    print(f"  Average loss: {loss:.4f}")

feature_time = time.time() - start_time

# Evaluate
feature_acc, _, _ = evaluate(model_feature, test_loader, device)

print(f"\n✅ FEATURE EXTRACTION Results:")
print(f"  Accuracy: {feature_acc:.4f}")
print(f"  Training time: {feature_time:.2f}s")

---

# Experiment 3: FINE-TUNING

**Approach**:
- Load pre-trained BERT
- **UNFREEZE** all layers
- Train the entire model (with low learning rate)

**Expected result**: Best performance (adapts pre-trained knowledge to task)

In [ ]:
print("=" * 60)
print("EXPERIMENT 3: FINE-TUNING")
print("=" * 60)

# Carica modello PRE-TRAINED
model_finetune = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
).to(device)

# TUTTI i parametri sono trainable (default)
print(f"\n🔢 Total parameters: {sum(p.numel() for p in model_finetune.parameters()):,}")
print(f"🔢 Trainable parameters: {sum(p.numel() for p in model_finetune.parameters() if p.requires_grad):,}")
print(f"   → 100% trainable!")

# Optimizer
optimizer = AdamW(model_finetune.parameters(), lr=2e-5)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# Training
start_time = time.time()
finetune_losses = []

for epoch in range(epochs):
    print(f"\n📍 Epoch {epoch + 1}/{epochs}")
    loss = train_epoch(model_finetune, train_loader, optimizer, scheduler, device)
    finetune_losses.append(loss)
    print(f"  Average loss: {loss:.4f}")

finetune_time = time.time() - start_time

# Evaluate
finetune_acc, finetune_preds, true_labels = evaluate(model_finetune, test_loader, device)

print(f"\n✅ FINE-TUNING Results:")
print(f"  Accuracy: {finetune_acc:.4f}")
print(f"  Training time: {finetune_time:.2f}s")

---

# Comparison & Analysis

In [ ]:
# Tabella comparativa
import pandas as pd

results_df = pd.DataFrame({
    'Method': ['From Scratch', 'Feature Extraction', 'Fine-tuning'],
    'Accuracy': [scratch_acc, feature_acc, finetune_acc],
    'Training Time (s)': [scratch_time, feature_time, finetune_time],
    'Trainable Params': [
        sum(p.numel() for p in model_scratch.parameters() if p.requires_grad),
        sum(p.numel() for p in model_feature.parameters() if p.requires_grad),
        sum(p.numel() for p in model_finetune.parameters() if p.requires_grad)
    ]
})

results_df['Improvement vs Scratch'] = ((results_df['Accuracy'] / results_df['Accuracy'].iloc[0] - 1) * 100).round(2).astype(str) + '%'

print("\n" + "="*80)
print("COMPARISON TABLE")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

In [ ]:
# Visualizzazione 1: Accuracy Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot accuracy
methods = ['From Scratch', 'Feature\nExtraction', 'Fine-tuning']
accuracies = [scratch_acc, feature_acc, finetune_acc]
colors = ['#e74c3c', '#f39c12', '#27ae60']

bars = axes[0].bar(methods, accuracies, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Transfer Learning: Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylim([0, 1])
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
axes[0].legend()

# Aggiungi valori sopra le barre
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{acc:.3f}',
                ha='center', va='bottom', fontweight='bold')

# Bar plot training time
times = [scratch_time, feature_time, finetune_time]
bars = axes[1].bar(methods, times, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Training Time (seconds)', fontsize=12)
axes[1].set_title('Training Time Comparison', fontsize=14, fontweight='bold')

for bar, t in zip(bars, times):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{t:.1f}s',
                ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualizzazione 2: Training Loss over epochs
plt.figure(figsize=(10, 6))

epochs_range = range(1, epochs + 1)
plt.plot(epochs_range, scratch_losses, 'o-', label='From Scratch', color='#e74c3c', linewidth=2, markersize=8)
plt.plot(epochs_range, feature_losses, 's-', label='Feature Extraction', color='#f39c12', linewidth=2, markersize=8)
plt.plot(epochs_range, finetune_losses, '^-', label='Fine-tuning', color='#27ae60', linewidth=2, markersize=8)

plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Training Loss', fontsize=12)
plt.title('Training Loss Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report per Fine-tuning
print("\n" + "="*80)
print("CLASSIFICATION REPORT - Fine-tuning (best model)")
print("="*80)
print(classification_report(true_labels, finetune_preds, target_names=['Negative', 'Positive']))

---

# Key Takeaways

## Performance

1. **From Scratch**: Poorest performance
   - No prior knowledge
   - Must learn everything from limited data
   - Typically ~50-60% accuracy on small datasets

2. **Feature Extraction**: Better performance
   - Uses pre-trained BERT features
   - Fast to train (only classifier)
   - Good when you have very limited data
   - Typically ~75-85% accuracy

3. **Fine-tuning**: BEST performance
   - Adapts pre-trained knowledge to task
   - Higher computational cost
   - State-of-the-art results
   - Typically ~85-95% accuracy

## ⏱Training Time

- **From Scratch**: Slowest (many parameters to learn)
- **Feature Extraction**: FASTEST (only classifier)
- **Fine-tuning**: Moderate (all parameters, but starts from good initialization)

## Data Efficiency

Transfer learning (Feature Extraction & Fine-tuning) achieves good performance with **10x less data** compared to training from scratch!

## When to use what?

| Approach | When to use |
|----------|-------------|
| **From Scratch** | • Very specialized domain<br>• Plenty of labeled data<br>• Computational resources available |
| **Feature Extraction** | • Very limited data (< 1000 samples)<br>• Limited compute<br>• Need fast iteration |
| **Fine-tuning** | • Moderate data (1000-10000+ samples)<br>• Want best performance<br>• Have some compute |

---



---

# BONUS: Visualize Attention Weights

Let's see what "looks at" BERT when classifying a review.

In [ ]:
def get_attention_weights(model, text):
    """Estrae attention weights per un testo"""
    model.eval()
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128).to(device)

    with torch.no_grad():
        outputs = model.bert(**inputs, output_attentions=True)

    # Prendi l'attention dell'ultimo layer, primo head
    attention = outputs.attentions[-1][0, 0].cpu().numpy()  # [seq_len, seq_len]
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    return attention, tokens

# Esempio: analizza una review positiva
sample_text = "This movie was absolutely fantastic! The acting was superb and the plot kept me engaged throughout."

attention, tokens = get_attention_weights(model_finetune, sample_text)

# Visualizza attention matrix
plt.figure(figsize=(12, 10))
sns.heatmap(
    attention,
    xticklabels=tokens,
    yticklabels=tokens,
    cmap='viridis',
    cbar_kws={'label': 'Attention Weight'}
)
plt.title('BERT Attention Weights (Last Layer, Head 0)', fontsize=14, fontweight='bold')
plt.xlabel('Tokens', fontsize=12)
plt.ylabel('Tokens', fontsize=12)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print(f"\nSample text: {sample_text}")
print(f"Tokens: {' '.join(tokens)}")

**Interpretation**:
- Lighter colors = higher attention
- See which words "looks atno" which other words
- Sentiment-related words ("fantastic", "superb") should have high attention

---

# 📚 References

- **BERT**: Devlin et al., "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding" (2018)
- **ULMFiT**: Howard & Ruder, "Universal Language Model Fine-tuning for Text Classification" (2018)
- **Transformers**: Vaswani et al., "Attention Is All You Need" (2017)
- **HuggingFace**: https://huggingface.co/transformers/

---

**👨‍🏫 Andrea Belli - Text Mining Course**